In [1]:
from pathlib import Path
import time
import pandas as pd
from ultralytics import YOLO

# ==================================================
# 1. ĐƯỜNG DẪN MODEL
# ==================================================
MODELS = {
    "YOLO11s": Path(
        r"C:\Users\anhla\Downloads\Web_flask\train_yolo11s\train-railway-yolo11s.pt"
    ),
    "YOLO26m": Path(
        r"C:\Users\anhla\Downloads\Web_flask\train-railway-yolo26m\runs\railway\weights\train-railway-yolo26m.pt"
    ),
}

# ==================================================
# 2. ĐƯỜNG DẪN DATASET
# ==================================================
DATASETS = {
    "roboflow_crossing": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing"
    ),
    "rcd_ground": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_ground"
    ),
    "rcd_elevated": Path(
        r"C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated"
    ),
}

# ==================================================
# 3. CẤU HÌNH ĐÁNH GIÁ
# ==================================================
IMGSZ = 640
BATCH = 16
CONF = 0.25
IOU = 0.60

OUTPUT_DIR = Path(
    r"C:\Users\anhla\Downloads\Web_flask\model_comparison_results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ==================================================
# 4. TÌM FILE DATA.YAML
# ==================================================
def find_data_yaml(dataset_dir):
    for filename in ["data.yaml", "data.yml"]:
        yaml_path = dataset_dir / filename
        if yaml_path.is_file():
            return yaml_path

    candidates = (
        list(dataset_dir.rglob("data.yaml"))
        + list(dataset_dir.rglob("data.yml"))
    )

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"Không tìm thấy data.yaml trong {dataset_dir}"
    )


# ==================================================
# 5. KIỂM TRA ĐƯỜNG DẪN
# ==================================================
for model_name, model_path in MODELS.items():
    if not model_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy model {model_name}: {model_path}"
        )

DATA_YAMLS = {}

for dataset_name, dataset_dir in DATASETS.items():
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy dataset: {dataset_dir}"
        )

    DATA_YAMLS[dataset_name] = find_data_yaml(dataset_dir)

print("Các file YAML được sử dụng:")
for name, path in DATA_YAMLS.items():
    print(f"{name}: {path}")


# ==================================================
# 6. ĐÁNH GIÁ 2 MODEL TRÊN 3 DATASET
# ==================================================
results = []

for model_name, model_path in MODELS.items():

    print("\n" + "=" * 65)
    print(f"ĐANG LOAD MODEL: {model_name}")
    print("=" * 65)

    model = YOLO(str(model_path))

    for dataset_name, yaml_path in DATA_YAMLS.items():

        print(f"\nModel: {model_name}")
        print(f"Dataset: {dataset_name}")
        print("-" * 65)

        run_name = f"{model_name}_{dataset_name}"

        start = time.perf_counter()

        metrics = model.val(
            data=str(yaml_path),
            split="val",
            imgsz=IMGSZ,
            batch=BATCH,
            conf=CONF,
            iou=IOU,
            plots=True,
            project=str(OUTPUT_DIR),
            name=run_name,
            exist_ok=True,
            verbose=True,
        )

        elapsed = time.perf_counter() - start

        # Lấy metrics
        box = metrics.box

        result = {
            "Model": model_name,
            "Dataset": dataset_name,
            "mAP50-95": float(box.map),
            "mAP50": float(box.map50),
            "mAP75": float(box.map75),
            "Precision": float(box.mp),
            "Recall": float(box.mr),
            "Validation_time_sec": round(elapsed, 2),
        }

        results.append(result)

        print("\nKẾT QUẢ:")
        print(f"mAP50-95 : {box.map:.4f}")
        print(f"mAP50    : {box.map50:.4f}")
        print(f"mAP75    : {box.map75:.4f}")
        print(f"Precision: {box.mp:.4f}")
        print(f"Recall   : {box.mr:.4f}")
        print(f"Thời gian: {elapsed:.2f} giây")

        # Lưu tiến độ sau mỗi lượt
        pd.DataFrame(results).to_csv(
            OUTPUT_DIR / "comparison_progress.csv",
            index=False,
            encoding="utf-8-sig",
        )


# ==================================================
# 7. XUẤT BẢNG TỔNG HỢP
# ==================================================
df = pd.DataFrame(results)

summary_path = OUTPUT_DIR / "comparison_summary.csv"

df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 90)
print("BẢNG SO SÁNH TỔNG HỢP")
print("=" * 90)

print(df.to_string(index=False))

print("\nĐã lưu kết quả tại:")
print(summary_path)


# ==================================================
# 8. SO SÁNH MODEL THEO TỪNG DATASET
# ==================================================
print("\n" + "=" * 90)
print("MODEL TỐT HƠN THEO mAP50-95 TRÊN TỪNG DATASET")
print("=" * 90)

for dataset_name in DATASETS:

    subset = df[df["Dataset"] == dataset_name]

    if subset.empty:
        continue

    best = subset.loc[subset["mAP50-95"].idxmax()]

    print(
        f"{dataset_name}: {best['Model']} "
        f"(mAP50-95 = {best['mAP50-95']:.4f})"
    )

print(f"\nToàn bộ kết quả nằm trong: {OUTPUT_DIR}")

Các file YAML được sử dụng:
roboflow_crossing: C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\data.yaml
rcd_ground: C:\Users\anhla\Downloads\dataset_testv3\rcd_ground\data.yaml
rcd_elevated: C:\Users\anhla\Downloads\dataset_testv3\rcd_elevated\data.yaml

ĐANG LOAD MODEL: YOLO11s

Model: YOLO11s
Dataset: roboflow_crossing
-----------------------------------------------------------------
Ultralytics 8.4.165  Python-3.10.11 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1626.2442.6 MB/s, size: 369.0 KB)
val: Scanning C:\Users\anhla\Downloads\dataset_testv3\roboflow_crossing\labels.cache... 120 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.3s/it 10.1s.4ss
                   all        120       1138   

Ultralytics 8.4.165  Python-3.10.11 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from 'C:\Users\anhla\Downloads\Web_flask\models\train-railway-yolo11s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (18.3 MB)

ONNX: starting export with onnx 1.23.0 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success  3.6s, saved as 'C:\Users\anhla\Downloads\Web_flask\models\train-railway-yolo11s.onnx' (36.2 MB)

Export complete (4.1s)
Results saved to C:\Users\anhla\Downloads\Web_flask\models\train-railway-yolo11s.onnx
Predict:         yolo predict task=detect model=C:\Users\anhla\Downloads\Web_flask\models\train-railway-yolo11s.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\anhla\Downloads\Web_flask\models\train-railway-yolo11s.onnx imgsz=640   
Visualize:       https://netron.app
Chuyển đổi sang ONNX thành công!
